# Loan Default Prediction — Exploratory Data Analysis

**Objective:** Understand the structure, quality, and patterns in the Home Credit loan application dataset before feature engineering and modelling.

**Questions we want to answer:**
- How imbalanced is the default class?
- Which features have the most missing data?
- What applicant characteristics correlate most strongly with default?
- Are there data quality issues we need to handle?

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import missingno as msno
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:.2f}'.format)

PALETTE = ['#2196F3', '#F44336']
sns.set_theme(style='whitegrid', palette=PALETTE, font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

In [ ]:
df = pd.read_csv('../data/application_train.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Dataset Overview

In [ ]:
print('=== Dataset Summary ===')
print(f'Rows:        {df.shape[0]:,}')
print(f'Columns:     {df.shape[1]:,}')
print(f'Numeric:     {df.select_dtypes(include=np.number).shape[1]}')
print(f'Categorical: {df.select_dtypes(include="object").shape[1]}')
print(f'Memory:      {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

In [ ]:
df.dtypes.value_counts()

## 3. Target Distribution — Class Imbalance

In [ ]:
target_counts = df['TARGET'].value_counts()
target_pct = df['TARGET'].value_counts(normalize=True) * 100

print('Target Distribution:')
print(f'  No Default (0): {target_counts[0]:,}  ({target_pct[0]:.1f}%)')
print(f'  Default    (1): {target_counts[1]:,}  ({target_pct[1]:.1f}%)')
print(f'  Imbalance ratio: {target_counts[0]/target_counts[1]:.1f}:1')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Count
axes[0].bar(['No Default', 'Default'], target_counts.values, color=PALETTE)
axes[0].set_title('Target Class Counts')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# Proportion
axes[1].pie(target_pct.values, labels=['No Default', 'Default'],
            autopct='%1.1f%%', colors=PALETTE, startangle=90)
axes[1].set_title('Target Class Proportion')

plt.suptitle('Class Imbalance — Default Rate', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Observation:** The dataset is heavily imbalanced. We will need to address this during modelling using class weights or resampling (SMOTE).

## 4. Missing Values

In [ ]:
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': df.isnull().mean() * 100
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

print(f'Columns with missing values: {len(missing)} / {df.shape[1]}')
print(f'\nTop 20 by missing %:')
missing.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
top_missing = missing.head(20)
bars = ax.barh(top_missing.index[::-1], top_missing['missing_pct'][::-1], color='#90CAF9')
ax.axvline(x=50, color='red', linestyle='--', alpha=0.5, label='50% threshold')
ax.set_xlabel('Missing (%)')
ax.set_title('Top 20 Features by Missing Data', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Missingness matrix for top columns
high_missing_cols = missing[missing['missing_pct'] > 40].index.tolist()
msno.matrix(df[high_missing_cols].sample(500, random_state=42), figsize=(12, 4))
plt.title('Missingness Matrix — High Missing Columns', fontweight='bold')
plt.show()

## 5. Key Numeric Features

In [ ]:
key_numeric = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'
]

df[key_numeric].describe().T

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(key_numeric):
    for target_val, color, label in zip([0, 1], PALETTE, ['No Default', 'Default']):
        data = df[df['TARGET'] == target_val][col].dropna()
        # Cap extreme outliers for readability
        cap = data.quantile(0.99)
        data = data[data <= cap]
        axes[i].hist(data, bins=40, alpha=0.5, color=color, label=label, density=True)
    axes[i].set_title(col, fontsize=9)
    axes[i].legend(fontsize=7)
    axes[i].tick_params(labelsize=7)

plt.suptitle('Key Numeric Feature Distributions by Target', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. External Source Scores (EXT_SOURCE)

The `EXT_SOURCE_1/2/3` features are external credit scores — typically the strongest predictors in this dataset.

In [ ]:
ext_sources = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(ext_sources):
    for target_val, color, label in zip([0, 1], PALETTE, ['No Default', 'Default']):
        data = df[df['TARGET'] == target_val][col].dropna()
        axes[i].hist(data, bins=40, alpha=0.6, color=color, label=label, density=True)
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend()

plt.suptitle('External Credit Scores by Default Status', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Mean EXT_SOURCE scores by target
df.groupby('TARGET')[ext_sources].mean().T.rename(columns={0: 'No Default', 1: 'Default'})

## 7. Categorical Features

In [ ]:
cat_cols = ['CODE_GENDER', 'NAME_CONTRACT_TYPE', 'FLAG_OWN_CAR',
            'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
            'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE']

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    default_rate = df.groupby(col)['TARGET'].mean().sort_values(ascending=False)
    bars = axes[i].bar(default_rate.index, default_rate.values * 100,
                       color=sns.color_palette('Blues_r', len(default_rate)))
    axes[i].set_title(f'Default Rate by {col}', fontweight='bold', fontsize=9)
    axes[i].set_ylabel('Default Rate (%)')
    axes[i].tick_params(axis='x', rotation=30, labelsize=7)
    axes[i].yaxis.set_major_formatter(mtick.PercentFormatter())
    for bar in bars:
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                     f'{bar.get_height():.1f}%', ha='center', fontsize=7)

plt.suptitle('Default Rate by Categorical Features', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 8. Correlation with Target

In [ ]:
correlations = df.select_dtypes(include=np.number).corr()['TARGET'].drop('TARGET')
correlations = correlations.dropna().sort_values()

top_pos = correlations.tail(15)
top_neg = correlations.head(15)
top_corr = pd.concat([top_neg, top_pos])

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#F44336' if v > 0 else '#2196F3' for v in top_corr.values]
ax.barh(top_corr.index, top_corr.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlation with TARGET')
ax.set_title('Top 15 Positive & Negative Correlations with Default', fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Age & Employment Analysis

In [ ]:
# DAYS_BIRTH is negative (days before application)
df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365

# DAYS_EMPLOYED has anomalous value 365243 — flag it
df['EMPLOYED_ANOMALY'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
df['DAYS_EMPLOYED_CLEAN'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
df['EMPLOYMENT_YEARS'] = -df['DAYS_EMPLOYED_CLEAN'] / 365

print(f"DAYS_EMPLOYED anomaly (365243) count: {df['EMPLOYED_ANOMALY'].sum():,}")
print(f"Anomaly default rate: {df[df['EMPLOYED_ANOMALY']==1]['TARGET'].mean():.3f}")
print(f"Normal default rate:  {df[df['EMPLOYED_ANOMALY']==0]['TARGET'].mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Age by target
for target_val, color, label in zip([0, 1], PALETTE, ['No Default', 'Default']):
    axes[0].hist(df[df['TARGET'] == target_val]['AGE_YEARS'].dropna(),
                 bins=40, alpha=0.6, color=color, label=label, density=True)
axes[0].set_title('Age Distribution by Default Status', fontweight='bold')
axes[0].set_xlabel('Age (years)')
axes[0].legend()

# Employment years by target
for target_val, color, label in zip([0, 1], PALETTE, ['No Default', 'Default']):
    data = df[(df['TARGET'] == target_val) & (df['EMPLOYED_ANOMALY'] == 0)]['EMPLOYMENT_YEARS'].dropna()
    data = data[data <= 40]
    axes[1].hist(data, bins=40, alpha=0.6, color=color, label=label, density=True)
axes[1].set_title('Employment Years by Default Status', fontweight='bold')
axes[1].set_xlabel('Years Employed')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Credit-to-Income & Annuity Ratios

In [ ]:
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['CREDIT_TERM'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']

ratio_cols = ['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(ratio_cols):
    for target_val, color, label in zip([0, 1], PALETTE, ['No Default', 'Default']):
        data = df[df['TARGET'] == target_val][col].dropna()
        cap = data.quantile(0.99)
        data = data[data <= cap]
        axes[i].hist(data, bins=40, alpha=0.6, color=color, label=label, density=True)
    axes[i].set_title(col, fontweight='bold', fontsize=9)
    axes[i].legend(fontsize=8)

plt.suptitle('Financial Ratio Distributions by Default Status', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 11. Correlation Heatmap — Key Features

In [ ]:
heatmap_cols = [
    'TARGET', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'AGE_YEARS', 'EMPLOYMENT_YEARS', 'AMT_CREDIT', 'AMT_INCOME_TOTAL',
    'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO'
]

corr_matrix = df[heatmap_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlation Matrix — Key Features', fontweight='bold')
plt.tight_layout()
plt.show()

## 12. EDA Summary & Key Findings

| Finding | Implication |
|---|---|
| ~8% default rate — severe class imbalance | Use `class_weight='balanced'` or SMOTE during modelling |
| `EXT_SOURCE_1/2/3` show strong separation | Include all three; engineer mean/min/max combinations |
| `DAYS_EMPLOYED = 365243` anomaly | Flag as binary feature; impute the rest |
| Younger applicants default more | `AGE_YEARS` is a useful feature — bin into age groups |
| High credit-to-income ratio → higher default | Engineer ratio features in notebook 02 |
| Many columns >40% missing | Drop or flag columns with very high missingness |

**Next:** Feature engineering in `02_feature_engineering.ipynb`